# 04 – Bygg aggregert Gold-tabell

Gold-laget inneholder ferdig aggregerte, business-ready datasett. Her bygger vi `department_stats` som gir lønnsstatistikk per avdeling direkte fra Silver.

Egenskaper:
- **Spark SQL** for aggregering — lett å lese og vedlikeholde
- **Delta merge** for inkrementell oppdatering — ikke full refresh
- **Kolonnebeskrivelser** lagret i Delta-metadataen

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import DeltaTable

spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .appName("04-build-gold")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

SILVER_PATH = "s3a://silver/employees"
GOLD_PATH   = "s3a://gold/department_stats"

## 1. Aggreger Silver → Gold med Spark SQL

In [ ]:
spark.read.format("delta").load(SILVER_PATH).createOrReplaceTempView("silver_employees")

gold_df = spark.sql("""
    SELECT
        department,
        COUNT(*)              AS headcount,
        ROUND(AVG(salary), 2) AS avg_salary,
        MIN(salary)           AS min_salary,
        MAX(salary)           AS max_salary,
        SUM(salary)           AS total_payroll,
        current_timestamp()   AS _gold_updated_at
    FROM silver_employees
    GROUP BY department
""")

gold_df.orderBy("department").show()

## 2. Skriv til Gold med Delta merge (inkrementell)

Merge på `department` betyr at:
- Eksisterende avdelinger **oppdateres** hvis tallene har endret seg
- Nye avdelinger **legges til**
- Kjøringen er idempotent — ingen duplikater

In [ ]:
if DeltaTable.isDeltaTable(spark, GOLD_PATH):
    gold_table = DeltaTable.forPath(spark, GOLD_PATH)
    (
        gold_table.alias("existing")
        .merge(gold_df.alias("incoming"), "existing.department = incoming.department")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("Merge fullført")
else:
    gold_df.write.format("delta").save(GOLD_PATH)
    print("Gold-tabell opprettet")

spark.read.format("delta").load(GOLD_PATH).orderBy("department").show()

## 3. Dokumenter kolonner i Delta-metadataen

Delta lagrer kolonnebeskrivelser i `_delta_log/` som en del av tabellskjemaet. Verktøy som Trino, DuckDB og datakatalogen kan lese disse direkte.

In [ ]:
table_ref = f"delta.`{GOLD_PATH}`"

# Tabellbeskrivelse
spark.sql(f"""
    ALTER TABLE {table_ref}
    SET TBLPROPERTIES (
        'comment' = 'Department-level salary and headcount statistics derived from Silver employees table.'
    )
""")

# Kolonnebeskrivelser
column_comments = {
    "department":       "Department name",
    "headcount":        "Number of active employees in the department",
    "avg_salary":       "Average salary (USD) across all employees in the department",
    "min_salary":       "Lowest salary (USD) in the department",
    "max_salary":       "Highest salary (USD) in the department",
    "total_payroll":    "Sum of all salaries (USD) in the department",
    "_gold_updated_at": "Timestamp when this row was last refreshed in the Gold layer",
}

for col, comment in column_comments.items():
    spark.sql(f"ALTER TABLE {table_ref} ALTER COLUMN {col} COMMENT '{comment}'")

print("Metadata satt")

## 4. Verifiser metadata og historikk

In [ ]:
# Vis skjema med kolonnebeskrivelser
spark.sql(f"DESCRIBE TABLE {table_ref}").show(truncate=False)

In [ ]:
# Vis tabellhistorikk
DeltaTable.forPath(spark, GOLD_PATH) \
    .history() \
    .select("version", "timestamp", "operation", "operationMetrics") \
    .show(truncate=60)

## 5. Bevis inkrementell oppdatering

Legg til en ny ansatt i Silver og re-kjør Gold-aggregeringen. Kun den berørte avdelingen skal oppdateres.

In [ ]:
from datetime import date

# Legg ny ansatt til Silver
ny_ansatt = spark.createDataFrame(
    [(200, "Lars", "sales", 75000, date(2025, 2, 1))],
    ["id", "name", "department", "salary", "hire_date"]
).withColumn("_silver_updated_at", F.current_timestamp())

ny_ansatt.write.format("delta").mode("append") \
    .option("mergeSchema", "true").save(SILVER_PATH)

print("Sales FØR oppdatering:")
spark.read.format("delta").load(GOLD_PATH) \
    .filter(F.col("department") == "sales") \
    .select("department", "headcount", "avg_salary", "total_payroll") \
    .show()

In [ ]:
# Re-kjør aggregering og merge
spark.read.format("delta").load(SILVER_PATH).createOrReplaceTempView("silver_employees")
gold_df2 = spark.sql("""
    SELECT department, COUNT(*) AS headcount,
           ROUND(AVG(salary), 2) AS avg_salary, MIN(salary) AS min_salary,
           MAX(salary) AS max_salary, SUM(salary) AS total_payroll,
           current_timestamp() AS _gold_updated_at
    FROM silver_employees GROUP BY department
""")

gold_table = DeltaTable.forPath(spark, GOLD_PATH)
(
    gold_table.alias("existing")
    .merge(gold_df2.alias("incoming"), "existing.department = incoming.department")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("Sales ETTER oppdatering (headcount +1, avg/total endret):")
spark.read.format("delta").load(GOLD_PATH) \
    .filter(F.col("department") == "sales") \
    .select("department", "headcount", "avg_salary", "total_payroll") \
    .show()

In [ ]:
spark.stop()